# Wildfire Web Inference Pipeline Preparation
สมุดงานนี้ปรับปรุงตามแนวทางเพื่อนำไปใช้งานบน Web Backend โดยแบ่งออกเป็น 2 ส่วนหลัก:
1. **One-Time Setup:** สร้างและบันทึกข้อมูล `Baseline` (จากอดีต) และ `KMeans Model` สำหรับนำไปเตรียมไว้บน Server/Database
2. **Web Backend Pipeline:** ฟังก์ชัน `preprocess_for_inference` สำหรับรับข้อมูลสภาวะอากาศ/ดาวเทียมของ "1 เดือนล่าสุด" มาแปลงเป็นฟีเจอร์สำหรับโมเดล CatBoost

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

# Create folder for Web Asset
os.makedirs('../Web_Assets', exist_ok=True)

## Part 1: One-Time Setup For Prepare Database and Model

In [2]:
print("--- 1. Prepare Baseline Data ---")
# Load all historical data
# Read the raw dataset containing historical records into a pandas DataFrame.
df_raw = pd.read_csv('../Dataset/df_final_model.csv')

# Calculate Baseline (historical monthly averages for each province) to use for anomaly detection
# Group data by province ('NAME_1') and 'month', then calculate the mean for NDVI, soil moisture, and temperature to establish a baseline.
baseline_df = df_raw.groupby(['NAME_1', 'month'])[['ndvi', 'soil_moisture', 'temp']].mean().reset_index()

# Rename the columns to clearly indicate that these are baseline values.
baseline_df.columns = ['NAME_1', 'month', 'ndvi_base', 'moisture_base', 'temp_base']

# Save the Baseline to a CSV file to be imported into the web database
# Export the resulting baseline DataFrame to a CSV file, excluding the row indices.
baseline_df.to_csv('../Web_Assets/baseline_table.csv', index=False)
print("✅ Saved: Web_Assets/baseline_table.csv (This table can be directly inserted into the DB)")

--- 1. Prepare Baseline Data ---
✅ Saved: Web_Assets/baseline_table.csv (This table can be directly inserted into the DB)


In [3]:
# Import necessary libraries (assuming they are imported earlier in the script)
# from sklearn.preprocessing import StandardScaler
# from sklearn.cluster import KMeans
# import joblib

print("--- 2. Train and Save KMeans Model (Cluster ID) ---")
# Extract data to calculate the district-level average before training KMeans (to ensure clusters match the main model training)
# Define the numerical columns that will be used as features for clustering.
num_cols = ['ndvi', 'soil_moisture', 'temp', 'elev', 'slope']

# Group the raw data by province, district, and month, and calculate the mean for the numerical columns.
df_train_cluster = df_raw.groupby(['NAME_1', 'NAME_2', 'month'])[num_cols].mean().reset_index()

# Initialize the StandardScaler to normalize the features so they have a mean of 0 and a variance of 1.
scaler = StandardScaler()
# Fill any missing values (NaN) with 0, then fit the scaler and transform the data simultaneously.
X_cluster = scaler.fit_transform(df_train_cluster[num_cols].fillna(0))

# Initialize the KMeans algorithm to divide the data into 2 distinct clusters.
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
# Train the KMeans model using the standardized data.
kmeans.fit(X_cluster)

# Save the Scaler and KMeans Model for the web application to load and use
# Serialize and save the fitted scaler object to a file.
joblib.dump(scaler, '../Web_Assets/cluster_scaler.pkl')
# Serialize and save the trained KMeans model to a file.
joblib.dump(kmeans, '../Web_Assets/cluster_kmeans.pkl')
print("✅ Saved: ../Web_Assets/cluster_scaler.pkl and ../Web_Assets/cluster_kmeans.pkl")

--- 2. Train and Save KMeans Model (Cluster ID) ---
✅ Saved: ../Web_Assets/cluster_scaler.pkl and ../Web_Assets/cluster_kmeans.pkl


## Part 2: Web Backend Pipeline (Web Data Conversion)

In [4]:
# Import necessary libraries for numerical operations and data manipulation.
import numpy as np
import pandas as pd

def preprocess_for_inference(df_current, baseline_df, scaler, kmeans, feature_list_path='../Models/model_d_features.csv'):
    """
    Function for Web Backend:
    1. Receive the latest month's data (df_current)
    2. Receive the Baseline table (averages from Train data only) to calculate Anomaly
    3. Calculate Physics & Cyclical Features
    4. Perform Clustering
    5. Return only the columns required by the model
    """
    # Create an independent copy of the input DataFrame to avoid modifying the original data.
    df = df_current.copy()
    
    # --- Group A: Physics-based Feature Engineering ---
    # Calculate using the same formulas as during training. 
    # These derive meaningful environmental indices from raw sensor data.
    df['veg_stress'] = (df['swir1'] - df['nir']) / (df['swir1'] + df['nir'] + 1e-6)
    df['fire_weather_idx'] = df['temp'] * (1.0 - df['soil_moisture'].clip(0, 1))
    df['wind_speed'] = np.sqrt(df['wind_u']**2 + df['wind_v']**2)
    df['drought_proxy'] = (1 - df['ndvi'].clip(-1, 1)) * (1 - df['soil_moisture'].clip(0, 1))
    df['terrain_roughness'] = df['slope'] * np.log1p(df['elev'])
    df['hot_dry_stress'] = df['temp'] * df['veg_stress']

    # --- Group B: Historical Anomaly (using the pre-prepared Baseline) ---
    # baseline_df must contain ['NAME_1', 'month', 'ndvi_base', 'moisture_base', 'temp_base']
    # Merge the current data with historical baselines based on province and month.
    df = df.merge(baseline_df, on=['NAME_1', 'month'], how='left')
    
    # Calculate the deviations (anomalies) from the historical baseline, filling any NaNs with 0.
    df['ndvi_anomaly'] = (df['ndvi'] - df['ndvi_base']).fillna(0)
    df['moisture_anomaly'] = (df['soil_moisture'] - df['moisture_base']).fillna(0)
    df['temp_anomaly'] = (df['temp'] - df['temp_base']).fillna(0)
    
    # Drop baseline columns to keep the DataFrame clean as it was in training.
    cols_to_drop = ['ndvi_base', 'moisture_base', 'temp_base']
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

    # --- Group C: Month Encoding (Cyclical) ---
    # Convert the 'month' feature into cyclical sine and cosine components to capture seasonal continuity.
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    # --- Group D: Unsupervised Features (Clustering) ---
    try:
        # scaler.feature_names_in_ contains the feature names in the order used during fitting.
        cluster_feats = scaler.feature_names_in_.tolist()
    except AttributeError:
        # In case of an older scaler without feature_names_in_, manually specify to match training.
        # Assuming it is: ['elev', 'ndvi', 'soil_moisture', 'slope', 'temp']
        cluster_feats = ['elev', 'ndvi', 'soil_moisture', 'slope', 'temp'] 
    
    # Check if all these columns are present in df and fill missing values.
    X_cluster = df[cluster_feats].fillna(0)
    
    # Now the column order in X_cluster will match exactly what the scaler requires.
    # Scale the features and assign the predicted cluster ID to the DataFrame.
    X_scaled = scaler.transform(X_cluster)
    df['cluster_id'] = kmeans.predict(X_scaled).astype(str)

    # --- Group E: Column Alignment ---
    # Retrieve the feature names saved during training to order the columns correctly.
    try:
        # Load the exact feature list required by the model.
        expected_cols = pd.read_csv(feature_list_path)['feature'].tolist()
        
        # In case there are features missing in df_current (e.g., forgot NAME_1 or other columns)
        for col in expected_cols:
            if col not in df.columns:
                df[col] = 0  # Fill missing features with 0 (to prevent breaking the model)
                
        # Return the DataFrame with columns explicitly ordered to match the model's expected input.
        return df[expected_cols]
    except Exception as e:
        print(f"⚠️ Warning: Could not align features from CSV: {e}")
        return df